# CONFIRMACIÓN SALIDA LISTA DE SÍMBOLOS PARA REVERSIONES FUTUROS - BINANCE

In [15]:
# =========================================================
# LIBRERÍAS
# =========================================================
import requests
import os
import pandas as pd
import numpy as np
import ta
import json
from pathlib import Path

import time
from datetime import datetime, timezone, timedelta

import winsound
import time

In [16]:
# =========================================================
# CONFIGURACIÓN GLOBAL
# =========================================================
BINANCE_FUTURES_BASE = "https://fapi.binance.com"
# INTERVAL = "15m"                 # coherente con scanner 1H
# LIMIT = 120                     # ~15 días de contexto
# LOOKBACK = 120
# MAX_SYMBOLS_TRAIN = 540       # limitar universo de entrenamiento
# MIN_NOTIONAL_VOLUME = 10_000  # Sugerido 5_000_000

# AVG_LIMIT_LONG = 30    # límite de KPI AVG para reversión LONG 
# AVG_LIMIT_SHORT = 70    # límite de KPI AVG para reversión SHORT 

FILEPATH = "C:/Users/Lenovo S540/Python_scripts/trades.json"
HISTORYPATH = "C:/Users/Lenovo S540/Python_scripts/trades_history.json"

In [17]:
# =========================================================
# CONFIGURACIÓN ALARMAS
# =========================================================
TELEGRAM_BOT_TOKEN = "8550995951:AAF9-6mGbQyJZv1Pml4MKu8ybVzMC3h-7S4"
TELEGRAM_CHAT_ID = "1733579121"

def send_telegram_alert(message: str):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "Markdown"
    }
    try:
        requests.post(url, json=payload, timeout=5)
    except Exception as e:
        print(f"[WARN] Telegram alert failed: {e}")



def local_warning():
    frequency = 2500  # Set Frequency To 2500 Hertz
    duration = 1000   # Set Duration To 1000 ms == 1 second

    winsound.Beep(frequency, duration)
    print("Oportunidades de entrada encontradas!")
    
    # Example of beeping with time delays
    for i in range(3):
        winsound.Beep(500, 200) # Frequency 500 Hz, duration 200ms
        time.sleep(0.5)        # Pause for 0.5 seconds between beeps

In [18]:
# =========================================================
# OHLC
# =========================================================
def get_ohlc(
    symbol: str,
    interval: str = "1h",
    limit: int = 500,
    max_total: int | None = None
):
    """
    Descarga OHLCV desde Binance Futures (USDT-M).
    
    - Usa requests (sin API key)
    - Soporta paginación si max_total > limit
    - Devuelve DataFrame limpio y ordenado
    """

    url = f"{BINANCE_FUTURES_BASE}/fapi/v1/klines"
    all_data = []
    end_time = None

    max_total = max_total or limit

    while len(all_data) < max_total:
        fetch = min(1500, max_total - len(all_data))

        params = {
            "symbol": symbol,
            "interval": interval,
            "limit": fetch
        }

        if end_time is not None:
            params["endTime"] = end_time

        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()
            
            if not data:
                break

            all_data = data + all_data
            end_time = data[0][0] - 1  # ms, vela anterior

        except Exception:
            break

    if len(all_data) < 50:
        return None

    df = pd.DataFrame(
        all_data,
        columns=[
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "close_time",
            "quote_asset_volume",
            "num_trades",
            "taker_buy_base_volume",
            "taker_buy_quote_volume",
            "ignore"
        ]
    )

    # ---------- Tipos ----------
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)

    # ---------- Timestamps ----------
    df["time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)

    df = df[["time", "open", "high", "low", "close", "volume"]]
    df.sort_values("time", inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.dropna(inplace=True)

    if len(df) < 50:
        return None

    return df

In [19]:
def get_current_prices_func(symbols, timeout=5):
    """
    Obtiene precio actual (last price) para múltiples símbolos desde Binance.
    
    Parámetros:
        symbols (list[str]) : lista de símbolos (ej: ["BTCUSDT", "ETHUSDT"])
        timeout (int)       : timeout HTTP en segundos
    
    Retorna:
        dict -> { "BTCUSDT": 64231.12, ... }
        o None si ocurre error
    """
    url = f"{BINANCE_FUTURES_BASE}/fapi/v1/ticker/price"
    if not symbols:
        return {}

    try:
        response = requests.get(
            url,
            timeout=timeout
        )

        if response.status_code != 200:
            print(f"[PRICE ERROR] Status code: {response.status_code}")
            return None

        data = response.json()

        # Convertir a dict eficiente
        price_map = {item["symbol"]: float(item["price"]) for item in data}

        # Filtrar solo símbolos solicitados
        result = {}

        for s in symbols:
            if s in price_map:
                result[s] = price_map[s]
            else:
                print(f"[WARNING] Símbolo no encontrado: {s}")

        return result

    except requests.exceptions.RequestException as e:
        print(f"[CONNECTION ERROR] {e}")
        return None

    except Exception as e:
        print(f"[UNEXPECTED ERROR] {e}")
        return None


In [20]:
def df15_provider(symbol: str, limit: int = 120):
    """
    Provee dataframe OHLCV de 15m con indicadores técnicos
    necesarios para confirmación y monitoreo.

    Retorna:
        pd.DataFrame o None
    """

    df = get_ohlc(symbol, interval="15m", limit=limit)

    if df is None or len(df) < 50:
        return None

    df = df.sort_index()

    # ---------------- INDICADORES ---------------- #
    df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()

    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / (avg_loss + 1e-9)
    df["rsi"] = 100 - (100 / (1 + rs))

    ema_fast = df["close"].ewm(span=12, adjust=False).mean()
    ema_slow = df["close"].ewm(span=26, adjust=False).mean()

    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=9, adjust=False).mean()

    df["macd_hist"] = macd - macd_signal

    return df


In [21]:
def df1m_provider(
    symbol: str,
    limit: int = 50 # Límite mínimo habilitado en el API Binance es 50
):
    """
    Proveedor OHLC 1m para ejecución de trade.

    Uso exclusivo:
    - Verificación precisa de SL / TP / TP parcial
    - Precio más reciente disponible

    Retorna:
    - DataFrame ordenado por tiempo ascendente
    - Columnas mínimas: time, open, high, low, close
    """

    try:
        df = get_ohlc(
            symbol=symbol,
            interval="1m",
            limit=limit
        )

        if df is None or df.empty:
            return None

        # Normalización defensiva
        df = df.copy()

        # Asegurar tipos numéricos
        for col in ["open", "high", "low", "close"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df["time"] = pd.to_datetime(df["time"], utc=True)

        # Limpiar datos inválidos
        df = df.dropna(subset=["close", "high", "low"])

        if len(df) == 0:
            return None

        # Orden temporal estricto
        df = df.sort_values("time").reset_index(drop=True)

        return df[["time", "open", "high", "low", "close"]]

    except Exception as e:
        print(f"[WARN] df1m_provider {symbol}: {e}")
        return None


In [22]:
def calculate_position_plan(
    entry_price: float,
    atr_15m: float,
    direction: str,
    target_usd_tp_partial: float,
    tp_partial_mult: float,
    sl_mult: float,
    max_leverage: int = 30,
    min_capital: float = 100.0,
    max_capital: float = 300.0,
    max_risk_usd: float = 80.0
):
    """
    Calcula capital y leverage óptimos basados en ATR y objetivo USD.

    Retorna:
        dict
    """

    # --------------------------------------------------
    # Movimiento esperado (TP parcial)
    # --------------------------------------------------
    tp_partial_move = tp_partial_mult * atr_15m
    tp_partial_pct = tp_partial_move / entry_price

    if tp_partial_pct <= 0:
        return None

    # --------------------------------------------------
    # Exposición necesaria para lograr target USD
    # --------------------------------------------------
    exposure_needed = target_usd_tp_partial / tp_partial_pct

    # --------------------------------------------------
    # Stop Loss en USD por unidad de exposición
    # --------------------------------------------------
    sl_move = sl_mult * atr_15m
    sl_pct = sl_move / entry_price

    # Capital mínimo para no violar riesgo
    capital_by_risk = max_risk_usd / sl_pct

    # Capital final limitado
    capital = max(min_capital, min(capital_by_risk, max_capital))

    # Leverage requerido
    leverage = exposure_needed / capital
    leverage = min(max(1.0, leverage), max_leverage)

    # Exposición final
    exposure = capital * leverage

    return {
        "capital_usd": round(capital, 2),
        "leverage": round(leverage, 1),
        "exposure_usd": round(exposure, 2),
        "tp_partial_usd": round(exposure * tp_partial_pct, 2),
        "sl_usd": round(exposure * sl_pct, 2)
    }


Nivel 1 — Deterioro de contexto (15m)

Momentum se debilita (MACD hist slope)

Precio pierde EMA21 o pierde estructura

No es simplemente pullback normal

Nivel 2 — Ruptura micro (1m)

Break de estructura reciente (rolling 20)

Movimiento significativo vs ATR (evita ruido)

Nivel 3 — Confirmación de debilidad (vela)

Rechazo fuerte

Cuerpo débil en contra

Colapso de rango

In [23]:
def compute_atr_from_df(df: pd.DataFrame, period: int = 14):
    if len(df) < period + 1:
        return None

    df = df.copy()
    df["prev_close"] = df["close"].shift(1)

    tr = np.maximum.reduce([
        df["high"] - df["low"],
        (df["high"] - df["prev_close"]).abs(),
        (df["low"] - df["prev_close"]).abs()
    ])

    atr = pd.Series(tr).rolling(period).mean().iloc[-1]

    if pd.isna(atr) or atr <= 0:
        return None

    return float(atr)


In [24]:
def compute_macd_histogram(
    df: pd.DataFrame,
    fast: int = 12,
    slow: int = 26,
    signal: int = 9,
    price_col: str = "close"
) -> pd.Series:
    """
    Calcula histograma MACD clásico (MACD - Signal).

    Diseñado para:
    - Timing fino de entrada (5m)
    - Evaluar energía + aceleración
    - Sin repainting (solo usa datos cerrados)

    Retorna:
        pd.Series alineada con df.index
    """

    if df is None or len(df) < slow + signal + 5:
        return pd.Series(dtype="float64")

    price = df[price_col].astype(float)

    # =========================
    # EMAs
    # =========================
    ema_fast = price.ewm(span=fast, adjust=False).mean()
    ema_slow = price.ewm(span=slow, adjust=False).mean()

    # =========================
    # MACD
    # =========================
    macd = ema_fast - ema_slow
    signal_line = macd.ewm(span=signal, adjust=False).mean()

    # =========================
    # HISTOGRAMA
    # =========================
    macd_hist = macd - signal_line

    return macd_hist


In [25]:
def structural_exit_conditions(symbol, side, df1m, df15):

    import numpy as np

    if len(df1m) < 50 or len(df15) < 50:
        return False

    # ======================================================
    # 1️⃣ CONTEXTO 15M
    # ======================================================

    ema_fast_15 = df15["close"].ewm(span=21).mean()
    ema_slow_15 = df15["close"].ewm(span=50).mean()

    macd_hist_15 = compute_macd_histogram(df15)
    atr_15 = compute_atr_from_df(df15, 14)

    price_15 = df15["close"].iloc[-1]

    trend_bull = ema_fast_15.iloc[-1] > ema_slow_15.iloc[-1]
    trend_bear = ema_fast_15.iloc[-1] < ema_slow_15.iloc[-1]

    macd_deteriorating_long = macd_hist_15.iloc[-1] < macd_hist_15.iloc[-2]
    macd_deteriorating_short = macd_hist_15.iloc[-1] > macd_hist_15.iloc[-2]

    price_lost_ema = (
        price_15 < ema_fast_15.iloc[-1]
        if side == "LONG"
        else price_15 > ema_fast_15.iloc[-1]
    )

    level1_context_break = False

    if side == "LONG":
        if trend_bull and macd_deteriorating_long and price_lost_ema:
            level1_context_break = True
    else:
        if trend_bear and macd_deteriorating_short and price_lost_ema:
            level1_context_break = True

    # ======================================================
    # 2️⃣ RUPTURA MICRO 1M
    # ======================================================

    price_1m = df1m["close"].iloc[-1]

    recent_high = df1m["high"].rolling(20).max().iloc[-2]
    recent_low = df1m["low"].rolling(20).min().iloc[-2]

    if side == "LONG":
        micro_break = price_1m < recent_low
    else:
        micro_break = price_1m > recent_high

    # Filtro anti-ruido usando ATR
    prev_close_1m = df1m["close"].iloc[-2]
    move_size = abs(price_1m - prev_close_1m)

    meaningful_move = move_size > (0.25 * atr_15)

    level2_micro_break = micro_break and meaningful_move

    # ======================================================
    # 3️⃣ CONFIRMACIÓN VELA MICRO
    # ======================================================

    last = df1m.iloc[-1]

    candle_range = last.high - last.low
    if candle_range <= 0:
        return False

    body = abs(last.close - last.open)
    body_rel = body / candle_range

    upper_wick = last.high - max(last.close, last.open)
    lower_wick = min(last.close, last.open) - last.low

    atr_rel = atr_15 / max(price_1m, 1e-9)

    weak_candle = False

    if side == "LONG":

        rejection = upper_wick / candle_range > 0.6
        weak_body = body_rel < 0.25 and last.close < last.open
        volatility_collapse = candle_range < 0.4 * atr_rel * price_1m

        if rejection or weak_body or volatility_collapse:
            weak_candle = True

    else:

        rejection = lower_wick / candle_range > 0.6
        weak_body = body_rel < 0.25 and last.close > last.open
        volatility_collapse = candle_range < 0.4 * atr_rel * price_1m

        if rejection or weak_body or volatility_collapse:
            weak_candle = True

    # ======================================================
    # 4️⃣ DECISIÓN FINAL JERÁRQUICA
    # ======================================================

    # Caso fuerte: ruptura macro + ruptura micro
    if level1_context_break and level2_micro_break:
        return True

    # Caso anticipado: ruptura micro + confirmación de debilidad
    if level2_micro_break and weak_candle:
        return True

    return False


In [26]:
# ======================================
# LOG HISTÓRICO
# ======================================

def log_trade_result(history_path, trade_record):

    if not os.path.exists(history_path):
        with open(history_path, "w") as f:
            json.dump([], f)

    try:
        with open(history_path, "r") as f:
            history = json.load(f)
    except:
        history = []

    history.append(trade_record)

    with open(history_path, "w") as f:
        json.dump(history, f, indent=4)

In [27]:
# ======================================
# MONITOR DE TRADES PRINCIPAL
# ======================================

def monitor_trades_from_file(
    file_path,
    history_path,
    get_current_prices_func,
    df1m_provider,
    df15_provider,
    calculate_position_plan,
    structural_exit_conditions,
    monitor_duration_hours=2,
    check_interval_seconds=15,
    error_retry_seconds=30
):

    start_time = datetime.now(timezone.utc)
    end_monitor_time = start_time + timedelta(hours=monitor_duration_hours)

    print(f"[Monitor] Activo hasta {end_monitor_time.isoformat()}")

    while datetime.now(timezone.utc) < end_monitor_time:

        loop_start = time.time()

        try:

            if not os.path.exists(file_path):
                time.sleep(check_interval_seconds)
                continue

            with open(file_path, "r") as f:
                trades = json.load(f)

            if not trades:
                time.sleep(check_interval_seconds)
                continue

            symbols = list(trades.keys())
            current_prices = get_current_prices_func(symbols)

            if current_prices is None:
                time.sleep(error_retry_seconds)
                continue

            now_utc = datetime.now(timezone.utc)
            symbols_to_delete = []

            for symbol, trade in trades.items():

                # ==================================================
                # INICIALIZACIÓN DINÁMICA (PENDING → OPEN)
                # ==================================================

                if trade["status"] == "PENDING":

                    df15 = df15_provider(symbol)
                    if df15 is None:
                        continue

                    atr_15m = compute_atr_from_df(df15, 14)

                    plan = calculate_position_plan(
                        entry_price=trade["entry_price"],
                        atr_15m=atr_15m,
                        direction=trade["side"],
                        target_usd_tp_partial=trade["target_usd_tp_partial"],
                        tp_partial_mult=trade["tp_partial_mult"],
                        sl_mult=trade["sl_mult"],
                        max_leverage=trade.get("max_leverage", 30),
                        min_capital=trade.get("min_capital", 100.0),
                        max_capital=trade.get("max_capital", 300.0),
                        max_risk_usd=trade.get("max_risk_usd", 80.0)
                    )
                                           
                    if plan is None:
                        continue

                    # Guardar plan
                    trade.update(plan)
                    trade["atr_15m"] = atr_15m

                    # Derivar niveles de precio
                    sl_move = trade["sl_mult"] * atr_15m
                    tp_partial_move = trade["tp_partial_mult"] * atr_15m

                    if trade["side"] == "LONG":
                        trade["initial_stop_loss"] = trade["entry_price"] - sl_move
                        trade["stop_loss"] = trade["initial_stop_loss"]
                        trade["tp_partial"] = trade["entry_price"] + tp_partial_move
                        trade["tp_final"] = trade["entry_price"] + 2 * tp_partial_move
                        trade["break_even_trigger"] = trade["entry_price"] + 0.8 * tp_partial_move
                        trade["break_even_price"] = trade["entry_price"]
                    else:
                        trade["initial_stop_loss"] = trade["entry_price"] + sl_move
                        trade["stop_loss"] = trade["initial_stop_loss"]
                        trade["tp_partial"] = trade["entry_price"] - tp_partial_move
                        trade["tp_final"] = trade["entry_price"] - 2 * tp_partial_move
                        trade["break_even_trigger"] = trade["entry_price"] - 0.8 * tp_partial_move
                        trade["break_even_price"] = trade["entry_price"]

                    trade["partial_taken"] = False
                    trade["be_activated"] = False
                    trade["status"] = "OPEN"

                    #print("trades", trades)
                    if trades[symbol]["printed"]=="NO" and "stop_loss" in trades[symbol]:
                        msg = (
                            f"🟢 *TRADE ABIERTO*\n"
                            f"Time: `{datetime.now()}`\n"
                            f"Symbol: `{symbol}`\n"
                            f"Direction: *{trade["side"]}*\n"
                            f"Entry: `{round(trade["entry_price"],6)}`\n"
                            f"SL inicial: `{round(trade["stop_loss"],6)}`\n"
                            f"TP parcial: `{round(trade["tp_partial"],6)}`\n"
                            f"TP final: `{round(trade["tp_final"],6)}`\n"
                        )
                        send_telegram_alert(msg)
                        print(msg)
                        trade["printed"]=="YES"

                # ==================================================
                # SOLO OPEN
                # ==================================================

                if trade["status"] != "OPEN":
                    continue

                if symbol not in current_prices:
                    continue

                price = current_prices[symbol]
                side = trade["side"]

                entry = trade["entry_price"]
                initial_sl = trade["initial_stop_loss"]
                sl = trade["stop_loss"]

                tp_partial = trade["tp_partial"]
                tp_final = trade["tp_final"]
                be_trigger = trade["break_even_trigger"]
                be_price = trade["break_even_price"]

                partial_taken = trade.get("partial_taken", False)
                be_activated = trade.get("be_activated", False)
                trailing_enabled = trade.get("trailing_enabled", True)

                end_time = datetime.fromisoformat(
                    trade["end_time_utc"]
                ).replace(tzinfo=timezone.utc)

                df1m = df1m_provider(symbol)
                df15 = df15_provider(symbol)

                if df1m is None or df15 is None:
                    continue

                # ==================================================
                # FUNCIÓN DE CIERRE (USD BASED)
                # ==================================================

                def close_trade(reason, close_price):

                    exposure = trade["exposure_usd"]
                    sl_usd = trade["sl_usd"]

                    if side == "LONG":
                        pnl_pct = (close_price - entry) / entry
                    else:
                        pnl_pct = (entry - close_price) / entry

                    pnl_usd = exposure * pnl_pct
                    r_multiple = pnl_usd / sl_usd if sl_usd != 0 else 0

                    trade_record = {
                        "symbol": symbol,
                        "side": side,
                        "entry_price": entry,
                        "close_price": close_price,
                        "exposure_usd": exposure,
                        "capital_usd": trade["capital_usd"],
                        "leverage": trade["leverage"],
                        "pnl_usd": round(pnl_usd, 2),
                        "sl_usd": sl_usd,
                        "r_multiple": round(r_multiple, 4),
                        "close_reason": reason,
                        "opened_at_utc": trade["opened_at_utc"],
                        "closed_at_utc": now_utc.isoformat(),
                        "partial_taken": partial_taken,
                        "be_activated": be_activated
                    }

                    log_trade_result(history_path, trade_record)
                    symbols_to_delete.append(symbol)

                # ==================================================
                # EXPIRACIÓN
                # ==================================================
                if now_utc >= end_time:
                    close_trade("EXPIRED", price)
                    msg = f"🔴 {symbol} {datetime.now()}: Tiempo expirado para el trade"
                    send_telegram_alert(msg)
                    print(msg)
                    continue

                # ==================================================
                # SALIDA ESTRUCTURAL
                # ==================================================

                if structural_exit_conditions(symbol, side, df1m, df15):
                    msg = f"⚠️ {symbol} {datetime.now()}: Salida Estructural"
                    send_telegram_alert(msg)
                    print(msg)
                    # close_trade("STRUCTURAL_EXIT", price)
                    # continue

                # ==================================================
                # STOP LOSS
                # ==================================================

                if (side == "LONG" and price <= sl) or \
                   (side == "SHORT" and price >= sl):
                    close_trade("STOP", price)
                    msg = f"🔴 {symbol} {datetime.now()}: Stop Loss"
                    send_telegram_alert(msg)
                    print(msg)
                    continue

                # ==================================================
                # TP PARCIAL
                # ==================================================

                if not partial_taken:
                    if (side == "LONG" and price >= tp_partial) or \
                       (side == "SHORT" and price <= tp_partial):

                        trade["partial_taken"] = True
                        msg = f"🟢 {symbol} {datetime.now()}: Take Profit Parcial"
                        send_telegram_alert(msg)
                        print(msg)

                # ==================================================
                # BREAK EVEN
                # ==================================================

                if not be_activated:
                    if (side == "LONG" and price >= be_trigger) or \
                       (side == "SHORT" and price <= be_trigger):

                        trade["be_activated"] = True
                        trade["stop_loss"] = be_price

                        msg = f"🟡 {symbol} {datetime.now()}: Break Even activado"
                        send_telegram_alert(msg)
                        print(msg)

                # ==================================================
                # TRAILING
                # ==================================================

                if trailing_enabled and trade.get("be_activated", False):

                    trailing_distance = trade["sl_mult"] * trade["atr_15m"] * 0.8

                    if side == "LONG":
                        new_sl = round(price - trailing_distance, 5)
                        if new_sl > trade["stop_loss"]:
                            trade["stop_loss"] = new_sl

                            msg = f"🔁 {symbol} {datetime.now()}: Trailing-Stop Loss actualizado {new_sl}"
                            send_telegram_alert(msg)
                            print(msg)
                            
                    else:
                        new_sl = round(price + trailing_distance, 5)
                        if new_sl < trade["stop_loss"]:
                            trade["stop_loss"] = new_sl

                            msg = f"🔁 {symbol} {datetime.now()}: Trailing-Stop Loss actualizado {new_sl}"
                            send_telegram_alert(msg)
                            print(msg)

                # ==================================================
                # TP FINAL
                # ==================================================

                if (side == "LONG" and price >= tp_final) or \
                   (side == "SHORT" and price <= tp_final):
                    close_trade("TP_FINAL", price)

                    msg = f"🟣 {symbol} {datetime.now()}: Take Profit Final"
                    send_telegram_alert(msg)
                    print(msg)

            # ==================================================
            # LIMPIEZA
            # ==================================================

            for symbol in symbols_to_delete:
                trades.pop(symbol, None)

            with open(file_path, "w") as f:
                json.dump(trades, f, indent=4)

        except Exception as e:
            print(f"[Monitor Error] {e}")
            time.sleep(error_retry_seconds)
            continue

        elapsed = time.time() - loop_start
        sleep_time = max(0, check_interval_seconds - elapsed)
        time.sleep(sleep_time)

    msg = f"⏰ {datetime.now()}: [Monitor de Trades] Finalizado por límite de tiempo."
    send_telegram_alert(msg)
    print(msg)


In [ ]:
monitor_trades_from_file(
    FILEPATH,
    HISTORYPATH,
    get_current_prices_func,
    df1m_provider,
    df15_provider,
    calculate_position_plan,
    structural_exit_conditions,
    monitor_duration_hours=2,
    check_interval_seconds=15,
    error_retry_seconds=30
)

[Monitor] Activo hasta 2026-03-06T00:58:58.263133+00:00
🟢 *TRADE ABIERTO*
Time: `2026-03-05 17:58:59.019035`
Symbol: `VVVUSDT`
Direction: *LONG*
Entry: `6.147`
SL inicial: `6.038371`
TP parcial: `6.309943`
TP final: `6.472886`

